# Advisor Match Agent: the three-stage workflow

This notebook explains and exercises the repository's **current implementation**: flexible upload interpretation, deterministic advisor matching, conversational exception review, and verified workbook generation.

> It intentionally makes no model or external database call. Executable examples use the checked-in synthetic source and the same deterministic Python modules as the application. Internal scores appear only for developer education; the runtime model and workbook never receive them.

The central boundary is simple: **the model interprets bounded evidence and conducts the conversation; deterministic application code owns identity decisions and workbook mutations.**

## Learning goals and setup

By the end, you should be able to explain:

1. how headed, later-header, and headerless uploads become an exact `InputMapping`;
2. why mapping validation and the missing-firm checkpoint happen before reference retrieval;
3. how an opaque authoritative snapshot is persisted and reused by one match session;
4. how CRD, email, name, firm, city, state, ZIP, nicknames, and conflicts affect deterministic decisions;
5. how bounded review, explicit overrides, audit history, and workbook regeneration work.

From the repository root, run `uv sync --locked --all-groups`, open this notebook, and select that environment.

In [1]:
from __future__ import annotations

import ast
import inspect
import sys
import tempfile
import textwrap
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
import yaml
from openpyxl import load_workbook


def find_project_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for path in (candidate, *candidate.parents):
        if (path / 'general_agent' / 'agent.py').is_file():
            return path
    raise RuntimeError('Start Jupyter in the advisor-match-agent repository.')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/charlie/Repos/deepagents-agents/advisor-match-agent


## 1. Architecture and responsibility boundary

```mermaid
flowchart TD
    U["One uploaded CSV or XLSX"] --> P["Bounded raw profiler"]
    P --> A{"One clear interpretation?"}
    A -->|"no"| Q["Ask the user"]
    Q --> A
    A -->|"yes"| V["Validate exact sheet, header row, indexes, and headers"]
    V --> F{"Name rows missing firm and strong IDs?"}
    F -->|"yes"| C["Ask for corrected upload or explicit continue"]
    F -->|"no"| R["find_all_advisors"]
    C --> R
    R --> S[("Protected opaque reference snapshot")]
    V --> M["Deterministic matcher"]
    S --> M
    M --> D[("Corporation-scoped session and audit")]
    M --> W["Verified advisor_matches.xlsx"]
    D --> E["Bounded exception pages"]
    E --> X{"Explicit user decision?"}
    X -->|"yes"| D
    D --> W
```

The model never sees the complete upload, authoritative table, persisted decision collection, internal numeric scores, protected snapshot path, or workbook internals. Application code enforces corporation/conversation scope, path and hash integrity, row limits, policy, audit, and export verification.

### The narrow agent and its eight tools

`general_agent/agent.py` creates one sole-purpose Deep Agent. It mounts only the `advisor-match` skill, exposes no shell/network/arbitrary-write/subagent capability, and instructs the model to ask when an interpretation is ambiguous. The typed tools are the workflow boundary; deterministic modules remain independently testable.

In [2]:
from general_agent.advisor_tools import build_advisor_tools
from general_agent.agent import SYSTEM_PROMPT

factory_source = textwrap.dedent(inspect.getsource(build_advisor_tools))
factory_node = ast.parse(factory_source).body[0]
tool_rows = []
for node in factory_node.body:
    is_tool = isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and any(
        isinstance(decorator, ast.Name) and decorator.id == 'tool'
        for decorator in node.decorator_list
    )
    if is_tool:
        tool_rows.append({'tool': node.name, 'contract': ast.get_docstring(node)})

tool_catalog = pd.DataFrame(tool_rows)
assert len(tool_catalog) == 8
print('Prompt opening:')
print('\n'.join(SYSTEM_PROMPT.splitlines()[:14]))
tool_catalog

Prompt opening:
You are Advisor Match Agent. Your sole purpose is to match financial-advisor
rows from one uploaded CSV or XLSX against the authoritative advisor reference,
conduct a bounded conversational review, and create `/advisor_matches.xlsx`.

Use the discovered advisor-match skill for every matching or review request.
Interpret the upload like an analyst: inspect bounded raw rows, select exactly
one worksheet, decide whether and where a header exists, and construct a typed
mapping from exact column indexes and observed headers. Ask the user when more
than one interpretation is plausible. Always call the mapping-validation tool
before retrieving the authoritative advisor snapshot. If validation reports
name rows without a firm, valid CRD, or valid email, ask whether the user can
provide a corrected upload or explicitly wants to continue.

The model must never decide identities row by row. Use deterministic tools for


,tool,contract
0,inspect_advisor_upload,Inspect bounded raw rows and plausible header ...
1,validate_advisor_input,Validate one interpreted advisor upload and r...
2,find_all_advisors,Snapshot all authoritative advisors and retur...
3,create_advisor_match,Create a persisted deterministic advisor mat...
4,get_current_advisor_match,Return this conversation's latest persisted ad...
5,list_advisor_match_results,List bounded advisor match results with qualit...
6,propose_crd_match,Resolve a user-supplied CRD and propose it for...
7,apply_advisor_match_decisions,Apply explicit advisor match decisions and reg...


The tools are deliberately ordered around three stages:

1. `inspect_advisor_upload` and `validate_advisor_input`;
2. `find_all_advisors` and `create_advisor_match`;
3. `get_current_advisor_match`, `list_advisor_match_results`, `propose_crd_match`, and `apply_advisor_match_decisions`.

A new match session requires a fresh snapshot. Later review and manual-CRD resolution reuse the session's immutable snapshot.

## 2. Stage 1 — interpret the upload flexibly

The profiler reads raw physical rows with `header=None`. For each bounded sheet it returns physical row numbers, up to five plausible header candidates, exact column indexes and observed headers, small post-header samples, patterns, and a headerless view. Suggestions are clues—not authority. If multiple sheets, header rows, or meanings remain plausible, the agent asks the user.

A headed mapping uses a one-based `header_row` and exact zero-based `(index, header)` references. A headerless mapping uses `header_row=None` and `header=None`. The validator rechecks those references before loading any mapped rows and returns a fingerprint over the source bytes plus canonical mapping.

In [3]:
from general_agent.advisor_matching.input_loader import validate_and_load_input
from general_agent.advisor_matching.profiler import inspect_advisor_upload
from general_agent.advisor_matching.schemas import InputMapping
from general_agent.config import Settings

tutorial_settings = Settings(project_root=PROJECT_ROOT, model_name='tutorial:no-model')
later_header_path = PROJECT_ROOT / 'examples' / 'advisor-match' / 'preamble_and_header.xlsx'
later_profile = inspect_advisor_upload(later_header_path, tutorial_settings)
later_sheet = later_profile['sheets'][0]
pd.DataFrame(later_sheet['preview_rows'])

,row_number,values
0,1,"[Quarterly advisor export, , , ]"
1,3,"[Advisor Name, Organization, Town, Province]"
2,4,"[John Smith, Northstar Wealth Partners, Boston..."
3,5,"[Robert Mercer, Cedar Grove Advisory, Richmond..."


In [5]:
later_mapping = InputMapping.model_validate({
    'sheet_name': 'Advisors',
    'header_row': 3,
    'full_name': {'columns': [{'index': 0, 'header': 'Advisor Name'}]},
    'firm_name': {'columns': [{'index': 1, 'header': 'Organization'}]},
    'city': {'columns': [{'index': 2, 'header': 'Town'}]},
    'state': {'columns': [{'index': 3, 'header': 'Province'}]},
})
loaded = validate_and_load_input(later_header_path, later_mapping, max_rows=50_000)
assert [row[0] for row in loaded.rows] == [4, 5]
print('Validated columns:', loaded.columns)
print('Mapping fingerprint:', loaded.mapping_fingerprint)
print('Input summary:', loaded.summary.model_dump())
pd.DataFrame([{'physical_row': number, **mapped} for number, _, mapped in loaded.rows])

Validated columns: [{'index': 0, 'header': 'Advisor Name', 'label': 'Advisor Name'}, {'index': 1, 'header': 'Organization', 'label': 'Organization'}, {'index': 2, 'header': 'Town', 'label': 'Town'}, {'index': 3, 'header': 'Province', 'label': 'Province'}]
Mapping fingerprint: 478f6e18cd6edd8ac79d8b3f15c3f20a8ee21d1d908253b085c77fc68ef0e385
Input summary: {'data_row_count': 2, 'blank_row_count': 0, 'preamble_row_count': 2, 'missing_firm_row_count': 0, 'missing_firm_confirmation_required': False}


,physical_row,crd_number,first_name,last_name,full_name,firm_name,email,city,state,zip_code
0,4,,,,John Smith,Northstar Wealth Partners,,Boston,MA,
1,5,,,,Robert Mercer,Cedar Grove Advisory,,Richmond,VA,


### Headerless input and the missing-firm checkpoint

Generated labels such as `Column A` are preview/display labels only. Exact headerless bindings still contain `header=None`. Completely blank rows are skipped while physical source row numbers are retained. Preamble rows above a selected header are not data.

After validation, any row with a usable multi-token name but no normalized firm, valid CRD, or valid email triggers one conversational checkpoint. The user can provide a corrected upload or explicitly continue. The agent does not patch values conversationally.

In [6]:
headerless_path = PROJECT_ROOT / 'examples' / 'advisor-match' / 'headerless_advisors.csv'
headerless_mapping = InputMapping.model_validate({
    'header_row': None,
    'crd_number': {'columns': [{'index': 0, 'header': None}]},
    'first_name': {'columns': [{'index': 1, 'header': None}]},
    'last_name': {'columns': [{'index': 2, 'header': None}]},
    'firm_name': {'columns': [{'index': 3, 'header': None}]},
    'city': {'columns': [{'index': 4, 'header': None}]},
    'state': {'columns': [{'index': 5, 'header': None}]},
})
headerless_loaded = validate_and_load_input(headerless_path, headerless_mapping, max_rows=50_000)
assert headerless_loaded.columns[0]['header'] is None
print(headerless_loaded.columns)

with tempfile.TemporaryDirectory(prefix='advisor-match-preflight-') as directory:
    name_only_path = Path(directory) / 'name-only.csv'
    name_only_path.write_text('Name\nRobert Mercer\n', encoding='utf-8')
    name_only_mapping = InputMapping.model_validate({
        'full_name': {'columns': [{'index': 0, 'header': 'Name'}]},
    })
    preflight = validate_and_load_input(name_only_path, name_only_mapping, max_rows=100)
assert preflight.summary.missing_firm_confirmation_required is True
preflight.summary.model_dump()

[{'index': 0, 'header': None, 'label': 'Column A'}, {'index': 1, 'header': None, 'label': 'Column B'}, {'index': 2, 'header': None, 'label': 'Column C'}, {'index': 3, 'header': None, 'label': 'Column D'}, {'index': 4, 'header': None, 'label': 'Column E'}, {'index': 5, 'header': None, 'label': 'Column F'}]


{'data_row_count': 1,
 'blank_row_count': 0,
 'preamble_row_count': 0,
 'missing_firm_row_count': 1,
 'missing_firm_confirmation_required': True}

## 3. Stage 2 — match deterministically

Only after mapping clarification does the agent call `find_all_advisors`. The tool projects the complete authoritative source to `CRD_NUMBER, FIRST_NAME, LAST_NAME, FIRM_NAME, EMAIL, CITY, STATE, ZIP_CODE`, stores it outside the agent-visible workspace, and returns an opaque ID plus manifest. The snapshot is corporation-and-conversation scoped, hash-checked, immutable, and single-use for creating a session.

`create_advisor_match` revalidates the upload/mapping fingerprint and snapshot integrity, loads rows, runs policy version 2, persists structured decisions, and generates the workbook. Malformed individual values become warnings or row-level `No Match` reasons. Only structural input, mapping, limit, or reference-integrity problems stop the run.

In [11]:
from general_agent.advisor_matching.schemas import MASTER_COLUMNS
from general_agent.advisor_matching.source import SyntheticAdvisorReferenceSource

master_path = PROJECT_ROOT / 'general_agent' / 'advisor_matching' / 'data' / 'master_advisors.csv'
advisors = list(SyntheticAdvisorReferenceSource(master_path).iter_records())
assert 'STREET_ADDRESS' not in MASTER_COLUMNS
print('Projected authoritative columns:', MASTER_COLUMNS)
print('Validated synthetic advisors:', len(advisors))
pd.DataFrame([advisor.model_dump() for advisor in advisors]).head(6)

Projected authoritative columns: ('CRD_NUMBER', 'FIRST_NAME', 'LAST_NAME', 'FIRM_NAME', 'EMAIL', 'CITY', 'STATE', 'ZIP_CODE')
Validated synthetic advisors: 40


,crd_number,first_name,last_name,firm_name,email,city,state,zip_code
0,99000001,Avery,Stone,Northstar Wealth Partners,avery.stone@example.com,Boston,MA,02108
1,99000002,John,Smith,Northstar Wealth Partners,john.smith.northstar@example.com,Boston,Massachusetts,02108-1200
2,99000003,John,Smith,Harbor Advisory Group,john.smith.harbor@example.com,Cambridge,MA,02139
3,99000004,Jon,Smyth,Summit Ridge Advisors,jon.smyth@example.com,Denver,CO,80202
4,99000005,Elizabeth,Hart,Blue Oak Financial,elizabeth.hart@example.com,Philadelphia,PA,19106
5,99000006,Robert,Mercer,Cedar Grove Advisory,robert.mercer@example.com,Richmond,VA,23219


### Decision order and evidence gates

1. Exact CRD is decisive; conflicting values become warnings.
2. A unique normalized email is strong. A non-unique authoritative email is `Ambiguous Match`.
3. A usable name is a multi-token full name or both first and last name.
4. An exact name still needs independent exact/close firm or exact city+state support.
5. A fuzzy name needs exact firm or exact city+state support. A fuzzy name plus fuzzy firm cannot auto-match without exact city+state.
6. Strong firm or state conflicts block name-based automation. A city difference within the same state is weaker.
7. Nickname-only and name-only evidence may generate candidates but never auto-match.
8. ZIP is display-only context: it contributes no score, support, or conflict. Street address is outside the matching domain.
9. Plausible unresolved candidates become `Ambiguous Match`; insufficient/invalid rows become `No Match` with a row-level reason.

The internal score ranks candidates and enforces deterministic thresholds; it is not a probability and is excluded from agent and workbook payloads.

In [ ]:
from general_agent.advisor_matching import normalization as norm
from general_agent.advisor_matching.policy import (
    ACCEPTANCE_SCORE, MINIMUM_FIRM_SIMILARITY, MINIMUM_MARGIN,
    MINIMUM_NAME_SIMILARITY, PLAUSIBLE_SCORE, POLICY_VERSION,
    REVIEW_CANDIDATE_LIMIT, WEIGHTS,
)

normalization_examples = pd.DataFrame([
    {'field': 'crd', 'raw': ' 99000006.0 ', 'normalized': norm.crd(' 99000006.0 ')},
    {'field': 'email', 'raw': ' Robert.Mercer@Example.COM ', 'normalized': norm.email(' Robert.Mercer@Example.COM ')},
    {'field': 'person_name', 'raw': 'Dr. Róbert Mercer, Jr.', 'normalized': norm.person_name('Dr. Róbert Mercer, Jr.')},
    {'field': 'firm', 'raw': 'Morgan Stanley & Co., LLC', 'normalized': norm.firm('Morgan Stanley & Co., LLC')},
    {'field': 'state', 'raw': 'Massachusetts', 'normalized': norm.state('Massachusetts')},
    {'field': 'zip context', 'raw': '02108-1200', 'normalized': norm.zip_code('02108-1200')},
])
policy = {
    'version': POLICY_VERSION, 'weights': WEIGHTS,
    'acceptance_score': ACCEPTANCE_SCORE, 'plausible_score': PLAUSIBLE_SCORE,
    'minimum_name_similarity': MINIMUM_NAME_SIMILARITY,
    'minimum_firm_similarity': MINIMUM_FIRM_SIMILARITY,
    'minimum_margin': MINIMUM_MARGIN, 'candidate_limit': REVIEW_CANDIDATE_LIMIT,
}
documented_policy = yaml.safe_load((PROJECT_ROOT / 'skills' / 'advisor-match' / 'references' / 'matching-policy.yaml').read_text())
assert documented_policy['fuzzy']['weights'] == WEIGHTS
print(policy)
normalization_examples

### Worked decisions against the real synthetic source

The next cell calls the same `run_matching` function used by the workflow tool. Qualitative candidate evidence is visible; internal score fields remain available only on the in-process developer model and are automatically excluded by `model_dump`.

In [ ]:
from general_agent.advisor_matching.matcher import run_matching

MAPPED_FIELDS = (
    'crd_number', 'first_name', 'last_name', 'full_name', 'firm_name',
    'email', 'city', 'state', 'zip_code',
)


def tutorial_row(row_number: int = 2, **values: str):
    mapped = {field: '' for field in MAPPED_FIELDS}
    mapped.update(values)
    return row_number, dict(mapped), mapped


cases = {
    'exact CRD wins despite conflicts': {'crd_number': '99000006', 'full_name': 'Someone Else', 'email': 'other@example.com', 'state': 'CA'},
    'unique email with unknown CRD': {'crd_number': '99999999', 'email': ' ELIZABETH.HART@EXAMPLE.COM '},
    'exact name plus legal-suffix firm': {'full_name': 'Avery Stone', 'firm_name': 'Northstar Wealth Partners, LLC'},
    'fuzzy name plus exact firm/location': {'full_name': 'John Smyth', 'firm_name': 'Summit Ridge Advisors', 'city': 'Denver', 'state': 'Colorado'},
    'nickname plus support is review-only': {'full_name': 'Bob Mercer', 'firm_name': 'Cedar Grove Advisory', 'city': 'Richmond', 'state': 'VA'},
    'name-only is review-only': {'full_name': 'John Smith'},
    'ZIP cannot support a name': {'full_name': 'John Smith', 'zip_code': '02108'},
    'malformed CRD only': {'crd_number': '99-000-006'},
    'firm/location without identity': {'firm_name': 'Cedar Grove Advisory', 'city': 'Richmond', 'state': 'VA'},
    'unknown named advisor': {'full_name': 'Quinn Example', 'firm_name': 'Imaginary Finance', 'city': 'Albany', 'state': 'NY'},
}

summary_rows = []
for label, values in cases.items():
    decision = run_matching([tutorial_row(**values)], advisors)[0][0]
    summary_rows.append({
        'case': label, 'status': decision.status, 'rule': decision.rule_id,
        'matched_crd': decision.matched_advisor.crd_number if decision.matched_advisor else None,
        'candidate_crds': ', '.join(candidate.crd_number for candidate in decision.candidates),
        'warnings': '; '.join(decision.warnings),
    })
case_summary = pd.DataFrame(summary_rows)
assert case_summary.loc[0, 'rule'] == 'EXACT_CRD'
assert case_summary.loc[4, 'status'] == 'Ambiguous Match'
assert case_summary.loc[7, 'rule'] == 'MALFORMED_CRD'
case_summary

### Duplicate rows and row-level outcomes

Duplicate signatures use normalized CRD, email, name, firm, city, state, and ZIP. Duplicates remain separate decisions with distinct review item IDs and physical source rows; a shared duplicate-group marker is an audit flag, not a deduplication action.

In [ ]:
duplicate_decisions, duplicate_counts, duplicate_warnings = run_matching([
    tutorial_row(2, crd_number='99000006'),
    tutorial_row(3, crd_number='99000006'),
], advisors)
assert duplicate_decisions[0].review_item_id != duplicate_decisions[1].review_item_id
assert duplicate_decisions[0].duplicate_group == duplicate_decisions[1].duplicate_group
print(duplicate_counts.model_dump(), duplicate_warnings)
pd.DataFrame([{
    'source_row': item.source_row_number, 'review_item_id': item.review_item_id,
    'duplicate_group': item.duplicate_group, 'status': item.status,
} for item in duplicate_decisions])

## 4. Stage 3 — review exceptions conversationally

After matching, the agent reports the interpreted mapping and three counts. It pages `Ambiguous Match` first, then offers `No Match` pages grouped by reason; automated matches are shown only on request. Review responses contain physical source rows, candidate CRDs, and qualitative supporting/conflicting/context evidence—never numeric scores.

Only explicit choices mutate effective decisions: confirm a presented candidate, confirm No Match, or propose an exact user-supplied CRD and confirm that resolved record in a later turn. The store appends full before/after JSON in the same transaction as the session update, preserves `automated_status`, increments the revision, and regenerates/verifies the workbook. Approval may retain unresolved exceptions. Source-value corrections require a new upload/session; there is no conversational row editor or `reopen` action.

In [ ]:
from general_agent.advisor_tools import _normalize_status_filter

status_inputs = ['Matched', 'matched', 'Ambiguous Match', 'ambiguous_match', 'No Match', 'no_match', 'unmatched']
pd.DataFrame({
    'tool_input': status_inputs,
    'canonical_status': [_normalize_status_filter(value) for value in status_inputs],
})

## 5. Human-first, auditable workbook

Every session revision regenerates exactly four sheets: `Matched`, `Review Required`, `Original Input`, and `Run Summary`. `Matched` exposes 17 human-facing columns; `Review Required` exposes 18. Technical IDs, rules, automated status, decision source, and duplicate group remain hidden at the right. Names and locations are combined for readability.

The generator adds filters, frozen headers, wrapping, alternating fills, status colors, bounded column widths at least as wide as headers, and capped result-row heights. CRD/ZIP and every user-controlled string are forced to text to prevent formula injection. Verification reopens the file, checks sheet order, rejects formulas, reconciles all decisions to Original Input, and validates all three persisted status counts before atomic publication.

In [ ]:
from general_agent.advisor_matching.matcher import run_matching
from general_agent.advisor_matching.schemas import ReferenceSnapshotManifest
from general_agent.advisor_matching.source import sha256_file
from general_agent.advisor_matching.workbook import verify_match_workbook, write_match_workbook

fixture_decisions, fixture_counts, fixture_warnings = run_matching(loaded.rows, advisors)
reference_manifest = ReferenceSnapshotManifest(
    reference_snapshot_id='ars_' + 'a' * 32,
    row_count=len(advisors), columns=list(MASTER_COLUMNS),
    source_kind='synthetic', schema_version='1',
    retrieved_at=datetime.now(UTC), sha256=sha256_file(master_path),
)

with tempfile.TemporaryDirectory(prefix='advisor-match-tutorial-') as directory:
    workbook_path = Path(directory) / 'advisor_matches.xlsx'
    write_match_workbook(
        workbook_path, session_id='ams_tutorial', decisions=fixture_decisions,
        counts=fixture_counts, mapping=later_mapping, input_summary=loaded.summary,
        source_name=later_header_path.name, source_sha256=loaded.source_sha256,
        reference=reference_manifest, policy_version=POLICY_VERSION,
    )
    verification = verify_match_workbook(
        workbook_path, expected_rows=len(loaded.rows), expected_counts=fixture_counts,
    )
    workbook = load_workbook(workbook_path, data_only=False)
    sheet_names = tuple(workbook.sheetnames)
    matched_headers = [cell.value for cell in workbook['Matched'][1]]
    visible_matched = [
        header for index, header in enumerate(matched_headers, start=1)
        if not workbook['Matched'].column_dimensions[workbook['Matched'].cell(1, index).column_letter].hidden
    ]
    formula_cells = [
        cell.coordinate for sheet in workbook.worksheets for row in sheet.iter_rows()
        for cell in row if cell.data_type == 'f'
    ]
    workbook.close()
assert len(visible_matched) == 17
assert not formula_cells
print('Counts:', fixture_counts.model_dump(), 'Warnings:', fixture_warnings)
print('Sheets:', sheet_names, 'Verification:', verification)

## 6. Concrete agent trace

| Step | Actor | Action | Model-visible result |
|---:|---|---|---|
| 1 | Agent | Read `advisor-match/SKILL.md` | Workflow instructions |
| 2 | Agent → profile | Inspect one upload | Bounded raw/header/headerless evidence |
| 3 | Agent | Choose a clear interpretation or ask | Candidate mapping only |
| 4 | Agent → validate | Bind exact sheet/index/header | Mapping, fingerprint, summary, missing-firm sample |
| 5 | User if needed | Correct upload or explicitly continue without firm | Explicit direction |
| 6 | Agent → find all | Retrieve authoritative source once | Opaque manifest, never rows/path |
| 7 | Agent → start | Pass mapping, fingerprint, snapshot ID | Session, mapping, counts, warnings, workbook path |
| 8 | Application | Match, persist, generate, verify | No model participation |
| 9 | Agent → list | Page ambiguous, then no-match exceptions | Bounded qualitative candidates |
| 10 | User | Select candidate/CRD or confirm no match | Explicit decision |
| 11 | Agent → apply | Submit typed choices/approval | Revised counts/status/workbook path |
| 12 | Application | Audit and regenerate/verify | No model participation |

## 7. Current boundaries and source map

The source adapter currently uses a 40-row synthetic CSV. Production Snowflake should implement the same projected schema and opaque manifest. At scale, replace the in-memory all-advisor fuzzy scan with stable server-side exact lookup and deterministic name blocking; do not move the master table into model context. Scores still require calibration against labeled data. Corporation IDs isolate storage but are not authentication, so both services remain loopback-only. Profile building remains intentionally unimplemented.

Key files:

- `general_agent/agent.py` — canonical three-stage prompt and narrow runtime;
- `general_agent/advisor_tools.py` — eight typed workflow/review tools;
- `general_agent/advisor_matching/schemas.py` — mapping, snapshot, decision, and review contracts;
- `profiler.py` / `input_loader.py` — bounded interpretation and exact validation;
- `normalization.py` / `policy.py` / `matcher.py` — deterministic identity policy;
- `general_agent/store.py` — snapshots, sessions, audits, and override proposals;
- `general_agent/advisor_matching/workbook.py` — four-sheet projection and verification;
- `skills/advisor-match/` — model-facing playbook and references;
- `tests/test_advisor_matching.py`, `test_advisor_tools.py`, and `test_advisor_workbook.py` — executable edge cases.

A useful next exercise is to add a labeled edge case, predict its decisive rule, support/conflict gates, candidate set, and row-level reason, then encode that prediction as a deterministic test before adjusting any threshold.